# Extract name, calcium level, potassium level, and sodium level.

### prompt used
```
You are a Python data-engineering assistant specializing in medical data
extraction and SQLite database operations.

## Context
I have a SQLite database `reports.db` containing a table `documents_cleaned`
with columns: id, filename, content.

The `content` column holds multi-page plain-text extracted from medical
laboratory reports. Each page follows this structure:

  --- Page N ---
  Laboratory Investigation Report
  Patient Name : <NAME> Centre : <CENTRE>
  ...
  <Test Name> [optional suffix]   <numeric result>   <unit>   <ref range>
  <method line>
  ...

## Task
Read every row from `documents` and extract the following fields:
- patient_name: Between "Patient Name : " and " Centre :"      |
- calcium    : Line starting with "Calcium" (may include a bracketed suffix e.g. "Calcium (Total)")     |
- potassium  : Line starting with "Potassium"                 |
- sodium     : Line starting with "Sodium"                    |

For calcium, potassium, and sodium — extract the numeric result and unit
only (e.g. "9.7 mg/dl", "5.03 mmol/L"). Ignore the reference range.

For calcium, potassium, and sodium — extract the range of normal values
 (e.g. "8.7 - 10.4", "3.5 - 5.1"). Use the reference range.


If a test is not present in a row, store NULL for that field.

## Output
Write extracted fields into a NEW table `documents_relevant` in the SAME
database `reports.db`. Recreate this table fresh on every run.

Schema:
  id            INTEGER PRIMARY KEY AUTOINCREMENT
  filename      TEXT
  patient_name  TEXT
  calcium       TEXT
  calcium_range TEXT
  potassium     TEXT
  potassium_range TEXT  
  sodium        TEXT
  sodium_range TEXT

## Implementation Rules
- Use Python's `re` module
- For lab values, match: keyword + optional bracketed suffix + numeric
  result + unit. Do NOT capture the reference range.
- For patient name, use a non-greedy match anchored between the two
  fixed tokens "Patient Name : " and " Centre :".
- Print a per-row extraction summary to stdout.
- The `documents` source table must NOT be modified.

## Sample Input (content of one row)
--- Page 1 ---
Laboratory Investigation Report
Patient Name : John Doe4 Centre : 5544 - Visit Health Pvt Ltd -Corporate
Age/Gender : 44 Y 10 M 10 D / M OP/IP No/UHID : //
Calcium (Total) 9.7 mg/dl 8.7 - 10.4
Arsenazo Colorimetric
Sodium 145.0 mmol/L 136 - 145
IMT
Potassium 5.03 mmol/L 3.5 - 5.1
IMT

## Expected Extracted Values
patient_name : John Doe4
calcium      : 9.7 mg/dl
calcium_range : 8.7 - 10.4
potassium    : 5.03 mmol/L
potassium_range : 3.5 - 5.1
sodium       : 145.0 mmol/L
sodium_range : 136 - 145


## Expected Console Output (per row)
[OK] id=1  file='rep1.pdf'
     patient_name : John Doe4
     calcium      : 9.7 mg/dl
     calcium_range : 8.7 - 10.4
     potassium    : 5.03 mmol/L
     potassium_range : 3.5 - 5.1
     sodium       : 145.0 mmol/L
     sodium_range : 136 - 145







In [2]:
"""
extract_relevant.py
-------------------
Reads `documents_clean` table from reports.db.
Extracts per row:
  - patient_name    : from "Patient Name : <n> Centre :"
  - calcium         : numeric result + unit
  - calcium_range   : reference range
  - potassium       : numeric result + unit
  - potassium_range : reference range
  - sodium          : numeric result + unit
  - sodium_range    : reference range

Sample line format:
  Calcium (Total) 9.7 mg/dl 8.7 - 10.4
  Sodium 145.0 mmol/L 136 - 145
  Potassium 5.03 mmol/L 3.5 - 5.1

Writes results to table `documents_relevant` in reports.db.
Table is recreated fresh on every run.
Source table `documents_clean` is NEVER modified.

Schema of documents_relevant:
  id               INTEGER PRIMARY KEY
  filename         TEXT
  patient_name     TEXT
  calcium          TEXT    e.g. "9.7 mg/dl"
  calcium_range    TEXT    e.g. "8.7 - 10.4"
  potassium        TEXT    e.g. "5.03 mmol/L"
  potassium_range  TEXT    e.g. "3.5 - 5.1"
  sodium           TEXT    e.g. "145.0 mmol/L"
  sodium_range     TEXT    e.g. "136 - 145"
"""

import sqlite3
import re
from pathlib import Path

DB_PATH      = "reports.db"
SOURCE_TABLE = "documents_cleaned"
TARGET_TABLE = "documents_relevant"

# ── Regex patterns ────────────────────────────────────────────────────────────

# Patient name: non-greedy match between fixed anchor tokens
RE_PATIENT = re.compile(
    r'Patient Name\s*:\s*(.+?)\s+Centre\s*:',
    re.IGNORECASE
)

# Lab value pattern captures three groups per test line:
#   group 1 → numeric result   e.g. "9.7"
#   group 2 → unit             e.g. "mg/dl"
#   group 3 → reference range  e.g. "8.7 - 10.4"
#
# Handles:
#   "Calcium (Total) 9.7 mg/dl 8.7 - 10.4"
#   "Sodium 145.0 mmol/L 136 - 145"
#   "Potassium 5.03 mmol/L 3.5 - 5.1"
def make_lab_re(keyword: str) -> re.Pattern:
    return re.compile(
        rf'^\s*{keyword}(?:\s*\([^)]*\))?\s+'  # keyword + optional (suffix)
        rf'(\d[\d.]*)'                           # group 1: numeric result
        rf'\s+'
        rf'(\S+)'                                # group 2: unit
        rf'\s+'
        rf'([\d.][\d.\s\-–]+)',                  # group 3: reference range
        re.IGNORECASE | re.MULTILINE
    )

RE_CALCIUM   = make_lab_re('Calcium')
RE_POTASSIUM = make_lab_re('Potassium')
RE_SODIUM    = make_lab_re('Sodium')


# ── Extraction helpers ────────────────────────────────────────────────────────

def extract_patient_name(content: str) -> str | None:
    m = RE_PATIENT.search(content)
    return m.group(1).strip() if m else None


def extract_lab(pattern: re.Pattern, content: str) -> tuple[str | None, str | None]:
    """Return (value_with_unit, reference_range) or (None, None) if not found."""
    m = pattern.search(content)
    if m:
        value = f"{m.group(1)} {m.group(2)}"
        rang  = m.group(3).strip()
        return value, rang
    return None, None


# ── Database helpers ──────────────────────────────────────────────────────────

def setup_target_table(conn: sqlite3.Connection):
    """Drop and recreate documents_relevant."""
    conn.execute(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
    conn.execute(f"""
        CREATE TABLE {TARGET_TABLE} (
            id               INTEGER PRIMARY KEY AUTOINCREMENT,
            filename         TEXT,
            patient_name     TEXT,
            calcium          TEXT,
            calcium_range    TEXT,
            potassium        TEXT,
            potassium_range  TEXT,
            sodium           TEXT,
            sodium_range     TEXT
        )
    """)
    conn.commit()


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    if not Path(DB_PATH).exists():
        print(f"ERROR: '{DB_PATH}' not found.")
        return

    # Open source read-only to guarantee no accidental writes
    src = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    rows = src.execute(
        f"SELECT id, filename, content FROM {SOURCE_TABLE}"
    ).fetchall()
    src.close()

    print(f"Source : {DB_PATH} → {SOURCE_TABLE}  ({len(rows)} row(s))")
    print(f"Target : {TARGET_TABLE}\n")

    # Open normally for writes to target table only
    conn = sqlite3.connect(DB_PATH)
    setup_target_table(conn)

    for row_id, filename, content in rows:
        if not content:
            print(f"  [SKIP] id={row_id} '{filename}' — empty content.")
            continue

        patient_name                = extract_patient_name(content)
        calcium,      calcium_range = extract_lab(RE_CALCIUM,   content)
        potassium, potassium_range  = extract_lab(RE_POTASSIUM, content)
        sodium,       sodium_range  = extract_lab(RE_SODIUM,    content)

        conn.execute(f"""
            INSERT INTO {TARGET_TABLE}
                (filename, patient_name,
                 calcium,   calcium_range,
                 potassium, potassium_range,
                 sodium,    sodium_range)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """, (filename, patient_name,
              calcium,   calcium_range,
              potassium, potassium_range,
              sodium,    sodium_range))
        conn.commit()

        print(f"  [OK] id={row_id}  file='{filename}'")
        print(f"       patient_name    : {patient_name}")
        print(f"       calcium         : {calcium}")
        print(f"       calcium_range   : {calcium_range}")
        print(f"       potassium       : {potassium}")
        print(f"       potassium_range : {potassium_range}")
        print(f"       sodium          : {sodium}")
        print(f"       sodium_range    : {sodium_range}\n")

    conn.close()
    print(f"Done. '{TARGET_TABLE}' written to '{DB_PATH}'.")


# ── Preview ───────────────────────────────────────────────────────────────────

def preview():
    conn = sqlite3.connect(DB_PATH)
    rows = conn.execute(f"""
        SELECT id, filename, patient_name,
               calcium,   calcium_range,
               potassium, potassium_range,
               sodium,    sodium_range
        FROM {TARGET_TABLE}
    """).fetchall()

    print(f"\n── {TARGET_TABLE} ({len(rows)} row(s)) ──────────────────────────────────")
    col = "{:<4}  {:<12}  {:<14}  {:<12}  {:<12}  {:<12}  {:<10}  {:<12}  {:<10}"
    print(col.format("id","filename","patient_name",
                     "calcium","ca_range","potassium","k_range","sodium","na_range"))
    print("─" * 100)
    for r in rows:
        print(col.format(*[str(v) for v in r]))
    conn.close()


if __name__ == "__main__":
    main()
    preview()

Source : reports.db → documents_cleaned  (2 row(s))
Target : documents_relevant

  [OK] id=1  file='rep1.pdf'
       patient_name    : John Doe4
       calcium         : 9.7 mg/dl
       calcium_range   : 8.7 - 10.4
       potassium       : 5.03 mmol/L
       potassium_range : 3.5 - 5.1
       sodium          : 145.0 mmol/L
       sodium_range    : 136 - 145

  [OK] id=2  file='rep2.pdf'
       patient_name    : John Doe4
       calcium         : None
       calcium_range   : None
       potassium       : None
       potassium_range : None
       sodium          : None
       sodium_range    : None

Done. 'documents_relevant' written to 'reports.db'.

── documents_relevant (2 row(s)) ──────────────────────────────────
id    filename      patient_name    calcium       ca_range      potassium     k_range     sodium        na_range  
────────────────────────────────────────────────────────────────────────────────────────────────────
1     rep1.pdf      John Doe4       9.7 mg/dl     8.7 - 